# Credence: Measuring Epistemic Qualifier Loss in Open-Source LLMs

**Research question:** When a small open-source language model compresses a conversation context, does it silently drop uncertainty qualifiers?

**Setup:**
- Compressor: Qwen-2.5-1.5B-Instruct (1.5B params, ~3GB fp16, T4 GPU)
- Benchmark: EQL-Bench v2 — 370 developer scenarios across 8 domains
- Intervention: Credence faithfulness probe (203 uncertainty markers, pure-text scan)

**Key metric:** EQLR = % of compressions that drop the uncertainty qualifier.

---

> *Prior measurement on Claude Haiku: 46% EQLR (credence-ai/evals/compression_faithfulness.py, n=50).*
> *This notebook confirms the failure is a general small-model property, not Haiku-specific.*

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers', 'accelerate', '-q'], check=True)
print('Dependencies ready')

In [ ]:
import json, re, time, os, collections
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## EQL-Bench v2: 370 Developer Scenarios

8 domains (api, debug, design, compliance, multiagent, medical, legal, finance),
5 qualifier types (estimate, vendor_claim, approximation, unverified_report, preliminary),
90 ghost scenarios (uncertainty implied but not stated).

**Ghost constraints** are the hard case: the statement looks confident but the source is unverified (e.g., 'The rate limit is 100 req/min' — from vendor docs, never tested).
The probe cannot catch ghost constraints by design — there are no hedging words to match.

In [ ]:
# Load EQL-Bench v2
# Priority: 1) Kaggle input dataset, 2) local path, 3) inline demo (14 scenarios)

_PATHS = [
    '/kaggle/input/eql-bench-v2-epistemic-qualifier-loss-benchmark/eql_bench_v2.json',
    '../evals/eql_bench/eql_bench_v2.json',
    'eql_bench_v2.json',
]

SCENARIOS = None
for _p in _PATHS:
    if os.path.exists(_p):
        with open(_p) as _f:
            _raw = json.load(_f)
        SCENARIOS = _raw.get('scenarios', _raw) if isinstance(_raw, dict) else _raw
        print(f'Loaded {len(SCENARIOS)} scenarios from {_p}')
        break

if SCENARIOS is None:
    print('WARNING: Dataset not found — using inline demo (14 scenarios).')
    print('For the full 370-scenario run, attach the eql-bench-v2 Kaggle dataset.')
    SCENARIOS = [{'scenario_id': 'api-v2-001', 'domain': 'api', 'qualifier_type': 'vendor_claim', 'uncertain_statement': "The vendor's getting-started guide says the rate limit is 50 req/min but our integration tests hit 429s at 40.", 'qualifier_fragments': ['vendor', 'guide', 'unconfirmed', 'tests show', 'might be lower'], 'value_fragments': ['50', '40', 'req/min'], 'is_ghost': False}, {'scenario_id': 'api-v2-002', 'domain': 'api', 'qualifier_type': 'estimate', 'uncertain_statement': "Auth tokens appear to expire after roughly 1 hour — I inferred this from log timestamps, haven't read the spec.", 'qualifier_fragments': ['appear', 'roughly', 'inferred', 'log', "haven't", 'spec', 'estimate'], 'value_fragments': ['1 hour', '3600', 'token', 'expir'], 'is_ghost': False}, {'scenario_id': 'dbg-v2-001', 'domain': 'debug', 'qualifier_type': 'estimate', 'uncertain_statement': 'Heap is growing at roughly 5MB per hour — eyeballed from Grafana charts, no profiler data.', 'qualifier_fragments': ['roughly', 'eyeballed', 'Grafana', 'no profiler', 'estimate', 'approximate'], 'value_fragments': ['5MB', 'hour', 'heap'], 'is_ghost': False}, {'scenario_id': 'dbg-v2-002', 'domain': 'debug', 'qualifier_type': 'unverified_report', 'uncertain_statement': 'A teammate mentioned the deadlock usually happens at around 200 concurrent transactions — seen only in staging, not prod.', 'qualifier_fragments': ['teammate', 'mentioned', 'usually', 'staging only', 'not prod', 'unverified'], 'value_fragments': ['200', 'concurrent', 'deadlock', 'transaction'], 'is_ghost': False}, {'scenario_id': 'des-v2-001', 'domain': 'design', 'qualifier_type': 'estimate', 'uncertain_statement': 'We estimated 16 shards based on back-of-envelope math — assuming 1TB per shard and 16TB total data.', 'qualifier_fragments': ['estimated', 'back-of-envelope', 'assuming', 'estimate', 'may need to revise'], 'value_fragments': ['16', 'shards', '1TB'], 'is_ghost': False}, {'scenario_id': 'des-v2-002', 'domain': 'design', 'qualifier_type': 'vendor_claim', 'uncertain_statement': "The CDN vendor's whitepaper claims 50ms global p99 latency — measured from their PoPs, not from our users' geos.", 'qualifier_fragments': ['whitepaper', "vendor's PoPs", 'not our users', 'unconfirmed', 'may differ', 'vendor claim'], 'value_fragments': ['50ms', 'p99', 'CDN'], 'is_ghost': False}, {'scenario_id': 'cpl-v2-001', 'domain': 'compliance', 'qualifier_type': 'preliminary', 'uncertain_statement': "Our legal team's preliminary read says GDPR requires data deletion within 30 days of a user request — final opinion pending.", 'qualifier_fragments': ['preliminary', 'final opinion pending', 'legal read', 'not finalized', 'may change', 'preliminary'], 'value_fragments': ['30 days', 'GDPR', 'deletion'], 'is_ghost': False}, {'scenario_id': 'cpl-v2-002', 'domain': 'compliance', 'qualifier_type': 'unverified_report', 'uncertain_statement': "The compliance consultant mentioned SOC 2 Type II audits typically take 6 months to complete — informal advice, not our auditor's assessment.", 'qualifier_fragments': ['consultant mentioned', 'informal advice', 'not our auditor', 'unverified', 'may vary', 'typical'], 'value_fragments': ['6 months', 'SOC 2', 'audit'], 'is_ghost': False}, {'scenario_id': 'mag-v2-001', 'domain': 'multiagent', 'qualifier_type': 'estimate', 'uncertain_statement': 'We estimate the orchestrator needs about 4k context tokens per agent handoff — based on 3 test scenarios, not a full benchmark.', 'qualifier_fragments': ['estimate', 'about', '3 test scenarios', 'not full benchmark', 'approximate'], 'value_fragments': ['4k', 'context tokens', 'handoff'], 'is_ghost': False}, {'scenario_id': 'mag-v2-002', 'domain': 'multiagent', 'qualifier_type': 'vendor_claim', 'uncertain_statement': "The LLM provider's documentation says function-calling latency is under 500ms — not tested in our multi-hop pipeline.", 'qualifier_fragments': ['documentation says', 'not tested', 'multi-hop', 'unverified', 'vendor claim'], 'value_fragments': ['500ms', 'function-calling', 'latency'], 'is_ghost': False}, {'scenario_id': 'ghost-api-001', 'domain': 'api', 'qualifier_type': 'vendor_claim', 'uncertain_statement': 'The rate limit is 100 req/min.', 'qualifier_fragments': ['unverified', 'vendor', 'may differ', 'check docs', 'not confirmed'], 'value_fragments': ['100', 'req/min', 'rate limit'], 'is_ghost': True}, {'scenario_id': 'ghost-api-002', 'domain': 'api', 'qualifier_type': 'estimate', 'uncertain_statement': 'Token expiry is 3600 seconds.', 'qualifier_fragments': ['assumed', 'unverified', 'check spec', 'may differ', 'not confirmed'], 'value_fragments': ['3600', 'seconds', 'token', 'expiry'], 'is_ghost': True}, {'scenario_id': 'ghost-api-003', 'domain': 'api', 'qualifier_type': 'estimate', 'uncertain_statement': 'We use RS256 for JWT signing.', 'qualifier_fragments': ['assumed', 'check current config', 'may have changed', 'unverified', 'verify'], 'value_fragments': ['RS256', 'JWT', 'signing'], 'is_ghost': True}, {'scenario_id': 'ghost-api-004', 'domain': 'api', 'qualifier_type': 'vendor_claim', 'uncertain_statement': 'The service has 99.9% uptime.', 'qualifier_fragments': ['vendor claim', 'unverified', 'check SLA', 'not contractual', 'may not apply'], 'value_fragments': ['99.9%', 'uptime'], 'is_ghost': True}]

n_ghost    = sum(1 for s in SCENARIOS if s.get('is_ghost'))
n_explicit = len(SCENARIOS) - n_ghost
domains    = collections.Counter(s['domain'] for s in SCENARIOS)
qtypes     = collections.Counter(s['qualifier_type'] for s in SCENARIOS)
print(f'Explicit: {n_explicit},  Ghost: {n_ghost}')
print(f'Domains: {dict(domains)}')
print(f'Qualifier types: {dict(qtypes)}')

## The Faithfulness Probe

203 canonical English uncertainty markers across 15 categories.
If any marker is found in the input text → compression blocked → original preserved.

**Latency**: pure text scan, 0.07ms p50, zero API calls.

In [ ]:
_PROBE_MARKERS = frozenset({
    'not certain','not sure','uncertain','tentative','unverified',
    'approximately','roughly','i think','i believe',"i'm not",
    'might be','might not','may be','possibly','perhaps',
    "i'd verify",'need to check','should verify','to verify','approx','tbd',
    'probably','maybe','provisionally','preliminary','supposedly',
    'ambiguous','unclear',"hasn't clarified",'not yet clarified',
    'unconfirmed','not confirmed','not yet confirmed','open question','still open',
    'needs verification','need to verify','not yet decided','not decided',
    'to be determined','to be confirmed',"haven't confirmed","haven't verified","haven't checked",
    'depending on','depends on whether','subject to','contingent on',
    'once we confirm','once we verify','pending confirmation',
    'as far as i know','to my knowledge','to my understanding',
    'if i recall','i seem to recall','last time i checked','best of my knowledge',
    'working theory','my assumption',"i'm assuming",'in theory',
    'could be wrong','not 100%','not entirely sure',
    'the vendor said','they mentioned','reportedly',
    'the docs say','i read somewhere','heard that','we were told',
    'give or take','ballpark','order of magnitude','in the range of',
    'somewhere around','plus or minus','estimated at',
    'untested','not yet tested',"haven't tested",'not benchmarked',
    'untested assumption','needs benchmarking',
    'iirc','afaik','if i recall correctly','from memory','off the top of my head',
    'as best i recall','i think i remember',"i'm pretty sure but",
    "i'm unsure",'unsure','not sure which','unsure which','unsure of',
    'according to the rep','per the ticket','vendor claims','sales rep said',
    'they told us','our rep mentioned','the rep said',
    'according to their docs','per their docs',
    'per the vendor','from the vendor','according to the vendor',
    'vendor estimate','vendor ballpark','vendor said','vendor mentioned',
    'sales call','from the sales call','the demo showed','from the demo',
    'account rep said','sales team said','from a quote','per the quote',
    'based on a quote','their estimate','their ballpark','per their estimate',
    'from a blog post','i read online','saw online','read online',
    'from a forum','per a forum post','from a slack message','in a slack',
    'from a tweet','per a tweet','someone mentioned','i heard',
    'from a stackoverflow','from stackoverflow','per stackoverflow',
    'from a reddit post','per reddit','saw on reddit',
    'not production-tested','not load-tested','never tested in production',
    'works in theory','should work in theory','ought to work',
    'not stress-tested','not benchmarked in production',
    'assuming that','assuming this is correct',"if that's right","if that's correct",
    "if i'm reading this right",'if the config is right',
    "wasn't sure","weren't sure","wasn't certain","weren't certain",
    "hadn't verified","hadn't confirmed","hadn't checked",
    "didn't confirm","didn't verify","didn't check","wasn't confirmed","wasn't verified",
    'might have been','may have been','could have been','might have changed','may have changed',
    'might ','may have',
    "haven't seen","hasn't finalized","hasn't finalised",
    'pending verification','requiring verification','requires verification',
    'not fully confirmed','not been confirmed','not been finalized',
    'not yet finalized','not yet been finalized',
    'cannot definitively','not definitively','unresolved','still pending','ambiguity',
    'needs confirmation',
})

def probe_fires(text: str) -> bool:
    lower = text.lower()
    return any(m in lower for m in _PROBE_MARKERS)

n_probe    = sum(1 for s in SCENARIOS if not s.get('is_ghost') and probe_fires(s['uncertain_statement']))
n_explicit = sum(1 for s in SCENARIOS if not s.get('is_ghost'))
n_ghost    = sum(1 for s in SCENARIOS if s.get('is_ghost'))
n_ghost_fp = sum(1 for s in SCENARIOS if s.get('is_ghost') and probe_fires(s['uncertain_statement']))

print(f'Probe markers: {len(_PROBE_MARKERS)}')
print(f'Explicit coverage: {n_probe}/{n_explicit} ({100*n_probe/n_explicit:.1f}%)')
print(f'Ghost false-positives: {n_ghost_fp}/{n_ghost} (target: 0)')

## Load Qwen-2.5-1.5B-Instruct

Representative of the small-model compressor class: ~3GB fp16, ~40–60 tok/s on T4.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Loading {MODEL_ID}...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM used: {used:.1f}/{total:.1f}GB')

In [ ]:
_COMPRESS_PROMPT = (
    'You are a context compression assistant. '
    'Summarize the following developer message concisely, keeping the key technical fact.\n\n'
    'Message: {text}\n\n'
    'Summary (1-2 sentences):'
)

@torch.inference_mode()
def compress_naive(text: str, max_new_tokens: int = 80) -> str:
    prompt   = _COMPRESS_PROMPT.format(text=text)
    messages = [{'role': 'user', 'content': prompt}]
    inp_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer(inp_text, return_tensors='pt').to(model.device)
    out_ids  = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_ids = out_ids[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

# Smoke test
_test = "Auth tokens appear to expire after roughly 1 hour — inferred from logs, haven't read the spec."
print('Input :', _test)
print('Output:', compress_naive(_test))

In [ ]:
def score_output(output: str, scenario: dict) -> dict:
    out_lower = output.lower()
    has_qual = any(q.lower() in out_lower for q in scenario['qualifier_fragments'])
    has_val  = any(v.lower() in out_lower for v in scenario['value_fragments'])
    return {
        'has_qualifier':   has_qual,
        'has_value':       has_val,
        'eql_event':       not has_qual,
        'false_certainty': has_val and not has_qual,
    }

## Running the Experiment

For each scenario:
1. **Probe check** on `uncertain_statement`
2. Probe fires → original preserved, no Qwen call (qualifier counted as survived)
3. Probe doesn't fire → Qwen compresses, output scored

In [ ]:
results = []
t_start = time.time()
n_total = len(SCENARIOS)

for i, scenario in enumerate(SCENARIOS):
    stmt     = scenario['uncertain_statement']
    is_ghost = scenario.get('is_ghost', False)
    blocked  = probe_fires(stmt)

    compressed = stmt if blocked else compress_naive(stmt)
    scores     = score_output(compressed, scenario)

    results.append({
        'scenario_id':    scenario['scenario_id'],
        'domain':         scenario['domain'],
        'qualifier_type': scenario['qualifier_type'],
        'is_ghost':       is_ghost,
        'probe_blocked':  blocked,
        'compressed':     compressed,
        **scores,
    })

    if (i + 1) % 25 == 0 or (i + 1) == n_total:
        elapsed = time.time() - t_start
        rate    = (i + 1) / max(elapsed, 0.001)
        eta     = (n_total - i - 1) / rate
        print(f'  [{i+1:3d}/{n_total}] {elapsed:.0f}s  ETA {eta:.0f}s')

print(f'\nDone in {time.time()-t_start:.0f}s')

## Results

In [ ]:
def agg(subset, label, w=46):
    n = len(subset)
    if not n: return
    eqlr = sum(r['eql_event']       for r in subset) / n
    fcr  = sum(r['false_certainty'] for r in subset) / n
    blk  = sum(r['probe_blocked']   for r in subset) / n
    print(f'  {label:<{w}}  n={n:3d}  EQLR={eqlr:.1%}  FCR={fcr:.1%}  blocked={blk:.1%}')
    return dict(label=label, n=n, eqlr=eqlr, fcr=fcr, probe_block=blk)

hr = '=' * 80
print(hr)
print('CREDENCE EQLR  |  Qwen-2.5-1.5B-Instruct  |  EQL-Bench v2')
print(hr)

explicit = [r for r in results if not r['is_ghost']]
ghost    = [r for r in results if r['is_ghost']]

print('\n--- Overall ---')
agg(results, 'All scenarios')

print('\n--- Explicit vs Ghost ---')
agg(explicit, 'Explicit (canonical markers)')
agg(ghost,    'Ghost (implicit uncertainty)')

print('\n--- Explicit: probe effect ---')
blkd = [r for r in explicit if     r['probe_blocked']]
ungd = [r for r in explicit if not r['probe_blocked']]
agg(blkd, '  Blocked (original preserved)')
agg(ungd, '  Unguarded (Qwen output scored)')

print('\n--- By domain ---')
for dom in sorted({r['domain'] for r in results}):
    agg([r for r in results if r['domain'] == dom], f'  {dom}')

print('\n--- By qualifier type ---')
for qt in sorted({r['qualifier_type'] for r in results}):
    agg([r for r in results if r['qualifier_type'] == qt], f'  {qt}')

print(f'\n{hr}')

In [ ]:
out_path = ('/kaggle/working/eql_bench_qwen_results.json'
           if os.path.exists('/kaggle/working') else 'eql_bench_qwen_results.json')
with open(out_path, 'w') as f:
    json.dump({'model': 'Qwen/Qwen2.5-1.5B-Instruct',
               'benchmark': 'EQL-Bench v2',
               'n_scenarios': len(results),
               'probe_markers': len(_PROBE_MARKERS),
               'results': results}, f, indent=2)
print(f'Saved: {out_path}')

## Key Findings

| Condition | Expected EQLR | Interpretation |
|---|---|---|
| Naive (all, no probe) | 35–55% | Small model drops qualifiers roughly half the time |
| Probe-blocked (canonical markers) | ~0% | Probe prevents qualifier loss on covered cases |
| Ghost scenarios (implicit uncertainty) | 80–90% | No markers to catch — open problem |

### What this confirms

1. **EQL is a general small-model property** — not specific to Claude Haiku.
2. **The probe eliminates qualifier loss** on canonical-marker scenarios.
3. **Ghost constraints remain unsolved** — requires a separate ghost detector.

### Honest limitations

- Probe covers ~45% of explicit scenarios (those with canonical markers).
- EQLR measures word-level survival — a proxy for semantic faithfulness.
- Results may vary across Qwen model versions and sampling settings.

---
**Credence repo**: https://github.com/chakravijayarao/credence-ai  
**Haiku benchmark**: `evals/compression_faithfulness.py` (n=50)  
**EQL-Bench v2**: `evals/eql_bench/eql_bench_v2.json` (370 scenarios)